# 3.2.1 — Projeção de Embeddings de **Documento** com BERT

**Aluna:** Bruna Soto Cardoso dos Santos  
**Corpus:** 35 documentos — Ciência e Tecnologia no Brasil (AD1)  
**Repositório:** https://github.com/brusoto/SRI_AD_TE

---
Gera embeddings BERT para cada documento e salva arquivos `.tsv` para o [TensorFlow Embedding Projector](https://projector.tensorflow.org/).

**Saídas** (salvas em `projecao/`):
- `tensors_documento.tsv` — vetores de embedding (768 dimensões)
- `metadata_documento.tsv` — rótulos dos documentos
- `config_documento.json` — configuração para o Projector

## Célula 1 — Instalação de dependências

In [ ]:
# ── Instala bibliotecas necessárias ──────────────────────────────────────────
!pip install transformers torch --quiet

## Célula 2 — Montar Google Drive e clonar repositório

In [ ]:
# ── Monta o Google Drive ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Caminho base do projeto no seu Drive ─────────────────────────────────────
# Ajuste se sua pasta tiver nome diferente
BASE_PATH = '/content/drive/MyDrive/Colab Notebooks/SRI'
DATA_PATH = os.path.join(BASE_PATH, 'data')
PROJ_PATH = os.path.join(BASE_PATH, 'projecao')

# Cria pasta projecao/ se não existir
os.makedirs(PROJ_PATH, exist_ok=True)

print(f'Base  : {BASE_PATH}')
print(f'Data  : {DATA_PATH}')
print(f'Saída : {PROJ_PATH}')

## Célula 3 — Baixar corpus do GitHub (se ainda não estiver no Drive)

In [ ]:
# ── Baixa documentos.csv do seu repositório pessoal ──────────────────────────
# Só executa se o arquivo ainda não existir localmente
CSV_FILE = os.path.join(DATA_PATH, 'documentos.csv')

if not os.path.exists(CSV_FILE):
    os.makedirs(DATA_PATH, exist_ok=True)
    # URL raw do seu repositório
    URL = 'https://raw.githubusercontent.com/brusoto/SRI_AD_TE/main/data/documentos.csv'
    !wget -q "{URL}" -O "{CSV_FILE}"
    print('documentos.csv baixado com sucesso.')
else:
    print('documentos.csv já existe no Drive.')

print(f'Arquivo: {CSV_FILE}')

## Célula 4 — Importações e carregamento do corpus

In [ ]:
import pandas as pd
import torch
import numpy as np
from transformers import BertTokenizer, BertModel

# ── Carrega o corpus ──────────────────────────────────────────────────────────
df = pd.read_csv(CSV_FILE)

# Garante colunas corretas (id, documento)
df.columns = [c.strip().lower() for c in df.columns]
if 'id' not in df.columns:
    df.insert(0, 'id', range(1, len(df)+1))

print(f'Documentos carregados: {len(df)}')
print(f'Colunas: {list(df.columns)}')
df.head(3)

## Célula 5 — Carregamento do modelo BERT multilingual

In [ ]:
# ── Modelo BERT multilingual (suporte a português) ────────────────────────────
MODEL_NAME = 'bert-base-multilingual-cased'

print(f'Carregando tokenizer: {MODEL_NAME} ...')
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

print(f'Carregando modelo: {MODEL_NAME} ...')
model = BertModel.from_pretrained(MODEL_NAME)
model.eval()

# Usa GPU se disponível
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f'Dispositivo: {device}')

## Célula 6 — Função para gerar embedding de documento (pooling [CLS])

In [ ]:
def get_document_embedding(text, tokenizer, model, device, max_length=512):
    """
    Gera embedding de um documento inteiro usando o token [CLS] do BERT.
    Trunca em max_length tokens se necessário.
    """
    inputs = tokenizer(
        text,
        return_tensors='pt',
        max_length=max_length,
        truncation=True,
        padding='max_length'
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    # Embedding do token [CLS] (índice 0 da última camada oculta)
    cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()
    return cls_embedding

## Célula 7 — Geração dos embeddings para todos os documentos

In [ ]:
# ── Gera embedding para cada documento ───────────────────────────────────────
embeddings = []
labels     = []

for _, row in df.iterrows():
    doc_id  = row['id']
    texto   = str(row['documento'])

    emb = get_document_embedding(texto, tokenizer, model, device)
    embeddings.append(emb)

    # Rótulo: primeiras 6 palavras do documento como legenda
    resumo = ' '.join(texto.split()[:6])
    labels.append(f'Doc_{doc_id}: {resumo}...')

    print(f'[{doc_id:02d}/{len(df)}] OK — {resumo[:40]}')

embeddings_np = np.array(embeddings)
print(f'\nShape dos embeddings: {embeddings_np.shape}')  # (35, 768)

## Célula 8 — Salvar arquivos TSV para o TensorFlow Embedding Projector

In [ ]:
# ── Salva tensors_documento.tsv ───────────────────────────────────────────────
tensor_path = os.path.join(PROJ_PATH, 'tensors_documento.tsv')
np.savetxt(tensor_path, embeddings_np, delimiter='\t')
print(f'Salvo: {tensor_path}')

# ── Salva metadata_documento.tsv ─────────────────────────────────────────────
meta_path = os.path.join(PROJ_PATH, 'metadata_documento.tsv')
with open(meta_path, 'w', encoding='utf-8') as f:
    f.write('id\ttema\n')
    for i, (row_id, label) in enumerate(zip(df['id'], labels)):
        tema = str(df.iloc[i]['documento']).split()[0:4]
        f.write(f'Doc_{row_id}\t{label}\n')
print(f'Salvo: {meta_path}')

## Célula 9 — Gerar config_documento.json para o Embedding Projector

In [ ]:
import json

# ── ATENÇÃO: substitua pelo URL raw do SEU repositório GitHub ────────────────
# Após fazer push dos arquivos TSV para a pasta projecao/ do seu repo,
# o URL seguirá o padrão abaixo:
GITHUB_RAW = 'https://raw.githubusercontent.com/brusoto/SRI_AD_TE/main/projecao'

config = {
    "embeddings": [
        {
            "tensorName": "Documentos — Ciência e Tecnologia BR",
            "tensorShape": [len(df), 768],
            "tensorPath": f"{GITHUB_RAW}/tensors_documento.tsv",
            "metadataPath": f"{GITHUB_RAW}/metadata_documento.tsv"
        }
    ]
}

config_path = os.path.join(PROJ_PATH, 'config_documento.json')
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print(f'Salvo: {config_path}')
print('\nConteúdo do config:')
print(json.dumps(config, indent=2, ensure_ascii=False))

## Célula 10 — Visualização rápida com matplotlib (opcional)

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# ── Reduz para 2D com PCA ─────────────────────────────────────────────────────
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(embeddings_np)

fig, ax = plt.subplots(figsize=(14, 9))

colors = cm.tab20(np.linspace(0, 1, len(df)))
for i, (x, y) in enumerate(coords):
    ax.scatter(x, y, color=colors[i], s=80, zorder=3)
    ax.annotate(
        f'Doc_{df.iloc[i]["id"]}',
        (x, y), fontsize=7, ha='center', va='bottom',
        xytext=(0, 5), textcoords='offset points'
    )

ax.set_title('Projeção PCA dos Embeddings de Documento (BERT)\nCorpus: Ciência e Tecnologia no Brasil',
             fontsize=13, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variância)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variância)')
ax.grid(True, alpha=0.3)
plt.tight_layout()

# Salva figura
fig_path = os.path.join(PROJ_PATH, 'grafico_documento.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Gráfico salvo: {fig_path}')

## Célula 11 — Link para o TensorFlow Embedding Projector

> **Passo a passo para visualizar:**
> 1. Faça push dos arquivos `projecao/` para o GitHub
> 2. Aguarde ~1 min para o GitHub processar os arquivos raw
> 3. Acesse o link gerado abaixo
> 4. Tire o print da visualização para o PDF da AD2

In [ ]:
PROJECTOR_URL = (
    'https://projector.tensorflow.org/?config='
    'https://raw.githubusercontent.com/brusoto/SRI_AD_TE/main/projecao/config_documento.json'
)
print('═' * 70)
print('LINK DO EMBEDDING PROJECTOR — DOCUMENTOS:')
print(PROJECTOR_URL)
print('═' * 70)
print('\nArquivos gerados em projecao/:')
for f in os.listdir(PROJ_PATH):
    size = os.path.getsize(os.path.join(PROJ_PATH, f))
    print(f'  {f:40s} {size/1024:.1f} KB')